In [1]:
!pip -q install google-genai gradio

In [1]:
import os, getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Cole sua GEMINI_API_KEY")


Cole sua GEMINI_API_KEY ········


In [5]:
import os
import gradio as gr
from google import genai
from google.genai import types

cliente = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MODELO = "gemini-2.5-flash"

SYSTEM_INSTRUCTION = """
Você é um minion.
Responda em Português do Brasil, com clareza e seja criativo com as respostas.
Se faltar informação, faça 1 pergunta curta antes de responder.
"""


def converter_historico_para_gemini(historico, mensagem_usuario):
    conteudo = []

    if historico is None:
        historico = []

    for item in historico:
        role = item["role"]
        texto = item["content"]

        if role == "user":
            role_gemini = "user"
        elif role == "assistant":
            role_gemini = "model"
        else:
            continue

        conteudo.append(
            types.Content(
                role=role_gemini,
                parts=[types.Part.from_text(text=texto[0].get('text', ''))]
            )
        )

    conteudo.append(
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=mensagem_usuario)]
        )
    )

    return conteudo


def chat_com_historico(mensagem_usuario, historico):
    conteudo = converter_historico_para_gemini(historico, mensagem_usuario)

    resposta = cliente.models.generate_content(
        model=MODELO,
        contents=conteudo,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0.6,
        ),
    )

    return resposta.text


with gr.Blocks(title="Super Chatbot Módulo 3") as demo:
    gr.Markdown("## Chatbot com histórico!")

    # Não coloque type="messages" nessa versão local
    chat = gr.Chatbot(height=500)

    msg = gr.Textbox(
        placeholder="Digite sua pergunta: ",
        show_label=False
    )

    clear = gr.Button("Limpar")

    def responder(mensagem_usuario, historico):
        if historico is None:
            historico = []

        resp = chat_com_historico(mensagem_usuario, historico)

        historico = historico + [
            {"role": "user", "content": mensagem_usuario},
            {"role": "assistant", "content": resp}
        ]

        return "", historico

    msg.submit(responder, [msg, chat], [msg, chat])
    clear.click(lambda: [], None, chat)

demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
